<a href="https://colab.research.google.com/github/annaannaR/NOTEBOOKS-FROM-SCHOOL/blob/main/AMS/Machine%20Learning/PART_2/MLP2_2_The_details_of_modeling_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
%matplotlib inline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df_iris = pd.read_csv("/content/drive/MyDrive/datasets/iris.csv")
features = df_iris.drop("species", axis=1)
labels = df_iris.species

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.dummy import DummyClassifier

#Overfitting

Overfitting means that your model interprets all kinds of noise in the training data as relevant, and will use this 'information' for future predictions. This often results in an overly complex model that can no longer generalize well to new situations.

The opposite can also happen: underfitting. Then your model ignores useful information, and as a result you cannot properly recognize the patterns in the data and use it for new predictions.

Different models have different levels of flexibility and complexity. But within a model, the level of complexity can be influenced by the settings of the hyperparameters.

In the case of k-NN, you can change the k to increase or decrease the complexity.

* lower k: more flexibility, and therefore a higher chance of overfitting
* higher k: less flexibility, and therefore a higher chance of underfitting

To find the best model, you'll want to play around with the settings to find the value that results in the best fit.

#Model complexity visualized

The code below shows a visualization of the complexity of a K-NN model. Don't mind the complexity of the code for a moment - it's taken from the scikitlearn website. The plot_contours function works like this:

1. it generates many different data points for different values of x1 and x2
2. it applies a model to all these points to make predictions
3. it colors the background based on the predictions to visualize what the model has learned

In [ ]:
def plot_contours(ax, clf, X):
    """Plot the decision boundaries for a classifier."""
    h = .02
    X0, X1 = X[:, 0], X[:, 1]
    x_min, x_max = X0.min() - 0.5, X0.max() + 0.5
    y_min, y_max = X1.min() - 0.5, X1.max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    out = ax.contourf(xx, yy, Z)
    return out

Now we can apply this visualization to KNN-models we train with different values of k, to see the difference.

Let's return to the dataset of the last assignment.

In [ ]:
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=500, shuffle=True, noise=0.4, random_state=1)
df_moons = pd.DataFrame(X, columns=["x1", "x2"])
df_moons["label"] = y
sns.lmplot(x="x1", y="x2", hue="label", data=df_moons, fit_reg=False)

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,5))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
for i, k in enumerate([1,30,150]):
    clf = KNeighborsClassifier(k)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    plot_contours(ax[i], clf, X)
    ax[i].scatter(X[:,0], X[:,1], c=y, cmap=plt.cm.coolwarm, s=20, edgecolors='k', alpha=0.7)
    ax[i].set_title("k = {}, Accuracy = {:.3f}".format(k, accuracy))

In the plots above you can see what happens at different sizes of k. At the far left we have a very low k. We see that a very complex model was created with many erratic lines and precise predictions. This is clearly a case of overfitting. When we apply this model to new values, the chance is quite small they behave exactly like this model.

On the far right you see that a lot of information has been thrown away. The model draws a very rough line through the two classes and leaves it at that.

The middle model looks like a compromise between the two options by including some of the special features in the pattern, but smoothing it out a bit more. You can see that the middle model also achieves the highest accuracy.

##Preprocessing

k-NN models sometimes have issues with "unequal axes" (features with values of different magnitude). We can do something about that so the features with higher values do not have a disproportionate influence on the results. For example, suppose we know people's annual income and their age. Annual income will usually consist of higher numbers than age, but we would not want annual income to weigh more heavily in the decision for that reason.

###Standardscaler

In Scikitlearn you can standardize your features to z-scores. That way you get rid of the uneven axes problem. This is a best practice when preprocessing your data for many algorithms.

###Transformers

The standardscaler is an example of a transformer in scikitlearn.
* Similar to predictors, transformers have a fit() function that calculates what the transformation parameters look like. For example, for the standardscaler, these are the mean and standard deviation of each column.
* To actually implement the transformation, use the transform() function.
* The fit_transform() function is a short way of calling fit() and transform() at once, in that order.

Note that transform() returns a numpy matrix instead of a pandas dataframe.
So we first have to explicitly put the result in a Dataframe again.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

In [ ]:
pd.DataFrame(features_scaled).describe()

We now see that all features have a mean and standard deviation of about 0 and 1, respectively.

Now we can split this dataset back into a train and test set.

**Question:** Can you think of possible problems using this approach?

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features_scaled, labels, test_size=0.4)

###Information leaks in the train_test_split

* during the train_test_split we have to be careful not to leak any information from the test data to the training data
* normalizing uses the entire dataset to determine the scaling parameters (mean and standard deviation are based on the entire dataset) - thus data from the test set suddenly contains information from the training set and vice versa
* the solution: we have to calculate the standardization factors based on the solely training part

So, the solution is to fit the scaler to the training set. And after that also apply it to the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.4)

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)